# First Agent

In [2]:
import asyncio
import os
import yfinance as yf
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown
from agents import Agent, Tool, Runner, function_tool

load_dotenv()  # Load environment variables from .env file
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))  # Initialize OpenAI

In [3]:
@function_tool
def get_stock_data(ticker: str)->dict:
    """
    Fetches real time stock data from yahoo finance
    Returns price, P/E ratio, market cap , revenue growth, and profit margin for the given ticker

    Args:
        ticker: Stock ticker symbol (e.g., AAPL for Apple Inc.)
    """
    stock = yf.Ticker(ticker)
    info = stock.info

    data = {
        "ticker": ticker,
        "current_price": info.get("currentPrice"),
        "pe_ratio": info.get("trailingPE"),
        "market_cap": info.get("marketCap"),
        "52_week_high": info.get("fiftyTwoWeekHigh"),
        "52_week_low": info.get("fiftyTwoWeekLow"),
        "revenue_growth": info.get("revenueGrowth"),
        "profit_margin": info.get("profitMargins")
    }

    return data

In [4]:
research_agent = Agent(
    name="StockResearchAgent",
    model="gpt-4o-mini",
    instructions="""
    You are a financial research assistant.
    
    ALWAYS use the get_stock_data tool before making any claims about a stock.
    Never guess or use training data for prices, ratios, or metrics.
    
    When analyzing a stock:
    1. Fetch real data first
    2. Cite the actual numbers in your analysis
    3. Be clear about what you know vs what requires further research
    """,
    tools=[get_stock_data]
)

In [8]:
result = await Runner.run(
    research_agent,
    "Can I invest in NVIDIA stock right now? What does the data say?",
)


In [11]:
print("Final Analysis:")
display(Markdown(result.final_output))

Final Analysis:


Here's the current data for NVIDIA (NVDA):

- **Current Price:** $208.58
- **P/E Ratio:** 31.99
- **Market Cap:** $5.05 trillion
- **52-Week High:** $236.54
- **52-Week Low:** $138.83
- **Revenue Growth:** 85.2%
- **Profit Margin:** 62.97%

### Analysis:

1. **Price vs. 52-Week Range:** The current price of $208.58 is below the 52-week high of $236.54 but well above the 52-week low of $138.83. This might indicate a potential for growth if the price approaches previous highs.

2. **P/E Ratio:** A P/E ratio of 31.99 indicates that investors might expect higher growth, which is common in tech companies like NVIDIA, especially given its revenue growth of 85.2%.

3. **Market Capitalization:** With a market cap of over $5 trillion, NVIDIA is a major player in the tech sector.

4. **Profitability:** A strong profit margin of 62.97% suggests that NVIDIA is efficient in managing its costs relative to its sales.

### Considerations:
- **Growth Potential:** With significant revenue growth, the company appears well-positioned in terms of potential future earnings.
- **Market Trends:** Investigate current market conditions and any recent news or developments that could impact NVIDIA.

### Conclusion:
Based on the data, NVIDIA is showing solid fundamentals, but investing decisions should also consider market conditions and personal financial goals. Further research into upcoming products, market demand, and tech sector performance would be prudent.